[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skyexry/urban-mobility-forecast/blob/main/notebooks/03_features_test.ipynb)

# 03 — Features Test
Validate the full feature pipeline: demand matrix → normalization → time features → sliding windows.
Uses the filtered 100-station dataset with input/output window = 72 hours.

In [ ]:
!git clone https://github.com/skyexry/urban-mobility-forecast.git 2>/dev/null || git -C urban-mobility-forecast pull

In [ ]:
import sys
import importlib.util
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')
sys.path.append('/content/urban-mobility-forecast')

Mounted at /content/drive


In [ ]:
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

graph_mod    = load_module('graph',    '/content/urban-mobility-forecast/preprocessing/graph.py')
features_mod = load_module('features', '/content/urban-mobility-forecast/preprocessing/features.py')

## 1. Load data (100 stations)

In [ ]:
df       = pd.read_parquet('/content/drive/MyDrive/citibike/hourly_demand_filtered.parquet')
stations = pd.read_parquet('/content/drive/MyDrive/citibike/stations_final.parquet')
df['hour'] = pd.to_datetime(df['hour'])


total_hours = df['hour'].nunique()
total_stations = df['start_station_id'].nunique()

expected = total_hours * total_stations
actual = len(df)
missing = expected - actual


print(f'Stations : {df["start_station_id"].nunique()}')
print(f'Rows     : {len(df):,}')
print(f'Date range: {df["hour"].min()} → {df["hour"].max()}')


print(f'Total hours    : {total_hours}')
print(f'Total stations : {total_stations}')
print(f'Expected rows  : {expected:,}')
print(f'Actual rows    : {actual:,}')
print(f'Missing (zeros): {missing:,}  ({missing/expected*100:.1f}%)')

Stations : 100
Rows     : 1,435,803
Date range: 2024-01-01 00:00:00 → 2025-12-31 23:00:00
Total hours    : 16808
Total stations : 100
Expected rows  : 1,680,800
Actual rows    : 1,435,803
Missing (zeros): 244,997  (14.6%)


## 2. Build demand matrix
Pivot sparse long-format data into dense `(T, N)` matrix. Missing hours filled with 0.

In [ ]:
station_ids = stations['start_station_id'].tolist()  # fixed node ordering

demand_matrix, hours = features_mod.build_demand_matrix(df, station_ids)
print(f'demand_matrix : {demand_matrix.shape}  (T x N)')
print(f'hours         : {len(hours)}, {hours[0]} → {hours[-1]}')

demand_matrix : (16808, 100)  (T x N)
hours         : 16808, 2024-01-01 00:00:00 → 2025-12-31 23:00:00


## 3. Normalize demand

log1p transform + MinMax scaling to [-1, 1]. Scaler is returned for inverse transform at inference.

**fit vs transform:**
- `fit` — learns parameters (min, max) from data. Must only be called on **training data**
- `transform` — applies learned parameters to any split (val, test, inference)
- `fit_transform` — does both at once (used here for pipeline validation only)

> ⚠️ **Data leakage note**: `normalize_demand` currently calls `fit_transform` on the full dataset,
> which means val/test statistics leak into the scaler. This is intentional here — this notebook
> only validates that the pipeline runs correctly end-to-end. In `04_train_eval.ipynb`, the scaler
> will be fit on the training split only, then applied separately to val and test.

In [ ]:
normalized, scaler = features_mod.normalize_demand(demand_matrix)
print(f'normalized : {normalized.shape}')
print(f'range      : [{normalized.min():.3f}, {normalized.max():.3f}]')

normalized : (16808, 100)
range      : [-1.000, 1.000]


## 4. Time features
Cyclic sin/cos encoding of hour-of-day, day-of-week, and day-of-year → 6 features per timestep.

In [ ]:
time_feats = features_mod.build_time_features(pd.Series(hours))
print(f'time_features : {time_feats.shape}  (T x 6)')

time_features : (16808, 6)  (T x 6)


## 5. Sliding windows
Input: past 72 hours. Output: next 72 hours. Window slides by 1 hour each step.

In [ ]:
x_demand, x_time, y = features_mod.build_sliding_windows(
    normalized, time_feats, input_window=72, output_window=72
)
print(f'\nx_demand : {x_demand.shape}  (samples, N, input_window, 1)')
print(f'x_time   : {x_time.shape}  (samples, input_window, 6)')
print(f'y        : {y.shape}  (samples, N, output_window, 1)')

Samples  : 16665
x_demand : (16665, 100, 72, 1)
x_time   : (16665, 72, 6)
y        : (16665, 100, 72, 1)

x_demand : (16665, 100, 72, 1)  (samples, N, input_window, 1)
x_time   : (16665, 72, 6)  (samples, input_window, 6)
y        : (16665, 100, 72, 1)  (samples, N, output_window, 1)


## 6. Sanity check

In [ ]:
# Verify no NaNs or Infs
assert not np.isnan(x_demand).any(), 'NaN in x_demand'
assert not np.isnan(y).any(),        'NaN in y'
assert not np.isnan(x_time).any(),   'NaN in x_time'
print('No NaNs or Infs — pipeline OK')

# Memory estimate
total_mb = (x_demand.nbytes + x_time.nbytes + y.nbytes) / 1e6
print(f'Total memory : {total_mb:.1f} MB')

No NaNs or Infs — pipeline OK
Total memory : 1017.5 MB
